## Usando o pandas para analisar o arquivo
---
* Caminho: C:\Users\joao.victor\MSGÁS\GEOP - Documentos\2026\Manutenção - 2026

In [ ]:
from calendar import month
from dbm import error
#instala as bibliotecas
!pip install pandas
!pip install openpyxl

In [ ]:
#importa bibliotecas
import pandas as pd

In [ ]:
camihho = r'C:\Users\joao.victor\MSGÁS\GEOP - Documentos\2026\Manutenção - 2026\Base de Dados IFS\BaseGEOP.xlsx'
try:
    df = pd.read_excel(camihho, sheet_name='IFS_TASK_CLOCKING')
    print("Carregado dataframe")
except Exception as e:
    print('Erro ao ler o excel', e)

## Tratamento de dados
* Apenas colunas, Nome em PT-BR - Nome em ING-IFSfrescos
   * N° Tarefa - TASK_SEQ
   * Descr da Tarefa - TASK_DESCRIPTION
   * Categoria de Registro - CLOCKING_CATEGORY (Viagem ou serviço)
   * Tipo de Reg de Horas - CLOCKING_TYPE (Registro de saída ou de entrada)
   * Org de Manut - ORGANIZATION_ID
   * Horário de Início - START_TIME
   * Hora Parada - STOP_TIME
   * Horas Trab - WORK_HOURS
   * ID Recurso - EMPLOYEE_ID

In [ ]:
essential_columns = [
  "TASK_SEQ",
  "TASK_DESCRIPTION",
  "CLOCKING_CATEGORY",
  "CLOCKING_TYPE",
  "START_TIME",
  "STOP_TIME",
  "WORK_HOURS",
  "ORGANIZATION_ID",
  "EMPLOYEE_ID"
]
df_clean = df[essential_columns]


In [ ]:
# mudando os nomes
df_clean.rename(columns={"TASK_SEQ" : "ID_tarefa",
  "TASK_DESCRIPTION": "Descricao",
  "CLOCKING_CATEGORY": "Tipo_temporal",
  "CLOCKING_TYPE": "Valida_registro",
  "START_TIME": "Tempo_inicio",
  "STOP_TIME": "Tempo_fim",
  "WORK_HOURS": "Horas_trabalhadas",
  "ORGANIZATION_ID": "ORG_manut",
  "EMPLOYEE_ID": "TOM"}, inplace=True)

In [ ]:
#Retirando dados que estão em execucao
df_clean.dropna(axis=0, how='any', inplace=True)

In [ ]:
# Formata os tempo_inicio e tempo_fim
#       função de formatação
def formata_data(d):
    #Variaveis
    r = str(d) #Variavel de retorno
    month_number = {
        "jan": "01",
        "feb": "02",
        "mar": "03",
        "apr": "04",
        "may": "05",
        "jun": "06",
        "jul": "07",
        "aug": "08",
        "sep": "09",
        "oct": "10",
        "nov": "11",
        "dec": "12"
    } # Dicionario conversor de mes de nome para numero
    r = r.strip().lower().split() #tirando espaços e deoxando tudo minusculo
    day = r[1].replace(",", " ").strip()
    month = month_number[r[0]]
    year = r[2][:4]
    hour = pd.to_timedelta(r[3]) + pd.to_timedelta("12:00:00") if r[4] == "pm" else pd.to_timedelta(r[3]) #horas "brasileiras"
    hour = str(hour)[-8:] # Transform o dado em string
    #Formato antes: May 5, 2026, 2:37:30 PM
    #Formato que deve ficar 05/05/2026 02:37:30
    return day + "/" + month + "/" + year + " " + hour
#       aplica função de formatação
df_clean["Tempo_inicio"] = df_clean["Tempo_inicio"].apply(formata_data)
df_clean["Tempo_fim"] = df_clean["Tempo_fim"].apply(formata_data)

In [119]:
help(pd.to_datetime)

Help on function to_datetime in module pandas:

to_datetime(
    arg: 'DatetimeScalarOrArrayConvertible | DictConvertible',
    errors: 'DateTimeErrorChoices' = 'raise',
    dayfirst: 'bool' = False,
    yearfirst: 'bool' = False,
    utc: 'bool' = False,
    format: 'str | None' = None,
    exact: 'bool | lib.NoDefault' = <no_default>,
    unit: 'str | None' = None,
    origin: 'str' = 'unix',
    cache: 'bool' = True
) -> 'DatetimeIndex | Series | DatetimeScalar | NaTType'
    Convert argument to datetime.

    This function converts a scalar, array-like, :class:`Series` or
    :class:`DataFrame`/dict-like to a pandas datetime object.

    Parameters
    ----------
    arg : int, float, str, datetime, list, tuple, 1-d array, Series, DataFrame/dict-like
        The object to convert to a datetime. If a :class:`DataFrame` is provided, the
        method expects minimally the following columns: :const:`"year"`,
        :const:`"month"`, :const:`"day"`. The column "year"
        must be 

In [121]:
# Formata colunas de tempo para deltatime
df_clean["Tempo_inicio"] = pd.to_datetime(df_clean["Tempo_inicio"], format="%d/%m/%Y %H:%M:%S")
df_clean["Tempo_fim"] = pd.to_datetime(df_clean["Tempo_fim"], format="%d/%m/%Y %H:%M:%S")

In [146]:

filtro_temporal = df_clean["Tempo_inicio"].between(pd.to_datetime("04/01/2026"), pd.to_datetime("04/30/2026"))
df_periodo = df_clean[filtro_temporal]

In [145]:
df_clean[filtro_temporal].to_excel("./data/meta_horas_ABRIL.xlsx")

In [ ]:
df_clean.head(10)